# Gold Daily Consumption, Temperature and Spot Prices

Create one analytical dataset for each Norwegian local date, price area and consumer group.

Daily Elhub consumption totals are joined with daily Frost temperature and available hourly spot prices. 

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from gold_layer.analytics import ingest

spark = (SparkSession.builder
         .appName("DAT535 Gold Analytics")
         .master("local[4]")
         .config("spark.driver.memory", "8g")
         .config("spark.sql.session.timeZone", "UTC")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

26/11/22 16:23:05 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/22 16:23:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/22 16:23:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/22 16:23:05 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## Build Gold analytics

In [2]:
result = ingest(
    silver_root=PROJECT_ROOT / "data/silver",
    gold_root=PROJECT_ROOT / "data/gold",
    start="2024-01-01",
    end="2026-01-01",
    spark=spark,
)

print(result)

{'dataset': 'daily_analytics', 'status': 'success', 'rows': 18275, 'gold_path': '/home/roland/UiS/rewrite/group14/data/gold/analytics/daily'}


## Inspect the output

In [3]:
analytics = spark.read.parquet(str(PROJECT_ROOT / "data/gold/analytics/daily"))
analytics.printSchema()
analytics.orderBy("date", "price_area", "consumption_group").show(10, truncate=False)

root
 |-- date: date (nullable = true)
 |-- price_area: string (nullable = true)
 |-- consumption_group: string (nullable = true)
 |-- consumption_kwh: decimal(28,6) (nullable = true)
 |-- avg_temperature_c: decimal(34,22) (nullable = true)
 |-- avg_spot_price_nok_kwh: double (nullable = true)
 |-- consumption_hour_count: integer (nullable = true)
 |-- consumption_is_complete: boolean (nullable = true)
 |-- station_count: long (nullable = true)
 |-- price_hour_count: integer (nullable = true)
 |-- price_is_complete: boolean (nullable = true)

+----------+----------+-----------------+---------------+-------------------------+----------------------+----------------------+-----------------------+-------------+----------------+-----------------+
|date      |price_area|consumption_group|consumption_kwh|avg_temperature_c        |avg_spot_price_nok_kwh|consumption_hour_count|consumption_is_complete|station_count|price_hour_count|price_is_complete|
+----------+----------+-----------------+----

## Validate row counts, keys and missing data

In [4]:
total = analytics.count()
expected = 731 * 5 * 5
print(f"Rows: {total:,} (expected {expected:,})")
assert total == expected, f"Unexpected row count: {total}"

key_duplicates = (analytics.groupBy("date", "price_area", "consumption_group")
                  .count().filter(F.col("count") > 1).count())
print("Duplicate date/area/group keys:", key_duplicates)
assert key_duplicates == 0

analytics.groupBy("price_area", "consumption_group").agg(
    F.count("*").alias("days"),
    F.countDistinct("date").alias("unique_days"),
).orderBy("price_area", "consumption_group").show(30, truncate=False)

analytics.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in analytics.columns
]).show(truncate=False)

missing = analytics.agg(
    F.sum(F.col("consumption_kwh").isNull().cast("int")).alias("consumption"),
    F.sum(F.col("avg_temperature_c").isNull().cast("int")).alias("temperature"),
    F.sum(F.col("avg_spot_price_nok_kwh").isNull().cast("int")).alias("spot_price"),
).first()


Rows: 18,275 (expected 18,275)
Duplicate date/area/group keys: 0
+----------+-----------------+----+-----------+
|price_area|consumption_group|days|unique_days|
+----------+-----------------+----+-----------+
|NO1       |cabin            |731 |731        |
|NO1       |household        |731 |731        |
|NO1       |primary          |731 |731        |
|NO1       |secondary        |731 |731        |
|NO1       |tertiary         |731 |731        |
|NO2       |cabin            |731 |731        |
|NO2       |household        |731 |731        |
|NO2       |primary          |731 |731        |
|NO2       |secondary        |731 |731        |
|NO2       |tertiary         |731 |731        |
|NO3       |cabin            |731 |731        |
|NO3       |household        |731 |731        |
|NO3       |primary          |731 |731        |
|NO3       |secondary        |731 |731        |
|NO3       |tertiary         |731 |731        |
|NO4       |cabin            |731 |731        |
|NO4       |household  

## Check price coverage and daylight saving time

A complete local day has 24 hours normally, 23 hours in spring and 25 hours in autumn. Some spot-price days may be incomplete because some were quarantined in Silver.

In [5]:
print("Spot-price completeness by area (days):")
(analytics.select("date", "price_area", "price_hour_count", "price_is_complete")
 .dropDuplicates(["date", "price_area"])
 .groupBy("price_area", "price_is_complete")
 .count().orderBy("price_area", "price_is_complete")
 .show(20, truncate=False))

print("DST transition dates:")
(analytics.filter(F.col("date").cast("string").isin(
    "2024-03-31", "2024-10-27", "2025-03-30", "2025-10-26"))
 .select("date", "price_area", "consumption_group", "consumption_hour_count",
         "price_hour_count", "price_is_complete")
 .filter(F.col("consumption_group") == "household")
 .orderBy("date", "price_area").show(25, truncate=False))

print("Incomplete or absent spot-price days:")
(analytics.select("date", "price_area", "price_hour_count", "price_is_complete")
 .dropDuplicates(["date", "price_area"])
 .filter(F.col("price_is_complete").isNull() | ~F.col("price_is_complete"))
 .orderBy("date", "price_area").show(40, truncate=False))

Spot-price completeness by area (days):
+----------+-----------------+-----+
|price_area|price_is_complete|count|
+----------+-----------------+-----+
|NO1       |NULL             |3    |
|NO1       |true             |728  |
|NO2       |NULL             |2    |
|NO2       |true             |729  |
|NO3       |NULL             |3    |
|NO3       |true             |728  |
|NO4       |NULL             |3    |
|NO4       |true             |728  |
|NO5       |NULL             |3    |
|NO5       |true             |728  |
+----------+-----------------+-----+

DST transition dates:
+----------+----------+-----------------+----------------------+----------------+-----------------+
|date      |price_area|consumption_group|consumption_hour_count|price_hour_count|price_is_complete|
+----------+----------+-----------------+----------------------+----------------+-----------------+
|2024-03-31|NO1       |household        |23                    |23              |true             |
|2024-03-31|NO2    

## Consumption totals

The analytical join must not change consumption quantities or duplicate consumption rows.

In [6]:
gold_consumption = spark.read.parquet(str(PROJECT_ROOT / "data/gold/consumption/daily"))
source_total = gold_consumption.agg(F.sum("consumption_kwh").alias("total")).first()["total"]
joined_total = analytics.agg(F.sum("consumption_kwh").alias("total")).first()["total"]
print("Consumption Gold total kWh:", source_total)
print("Analytical Gold total kWh:", joined_total)

incomplete_consumption = analytics.filter(~F.col("consumption_is_complete")).count()
print("Incomplete consumption rows:", incomplete_consumption)

Consumption Gold total kWh: 262407656724.770000
Analytical Gold total kWh: 262407656724.770000
Incomplete consumption rows: 0
